In [ ]:
15 50

In [ ]:
conda activate tensflow

pip install ray[tune]

pip install ipywidgets

In [ ]:
conda activate tensflow && jupyter lab --ip=0.0.0.0 --port=8888 --allow-root --no-browser --IdentityProvider.token=''

cd /home/oleg/projects/ML/Wildfire_Satelite && mlflow server --backend-store-uri mlruns --host 0.0.0.0 --port 5000

conda activate tensflow

jupyter lab --ip=0.0.0.0 --port=8888 --allow-root --no-browser --IdentityProvider.token=''

mlflow server --backend-store-uri mlruns --host 0.0.0.0 --port 5000

watch -n 1 nvidia-smi

sudo reboot now


In [3]:
'''
Проверенные параметры для YOLO-CLS:
results = model.train(
    # 🎯 ОСНОВНЫЕ
    data=IMAGE_PATH,
    epochs=10,
    imgsz=224,
    batch=16,
    device=0,
    
    # 🔧 ОПТИМИЗАТОР И LEARNING RATE
    lr0=0.01,           # Начальный learning rate
    lrf=0.01,           # Финальный learning rate (lr0 * lrf)
    momentum=0.937,      # SGD momentum
    weight_decay=0.0005, # L2 регуляризация
    warmup_epochs=3.0,   # Эпохи разогрева
    warmup_momentum=0.8, # Momentum при разогреве
    warmup_bias_lr=0.1,  # Learning rate для bias при разогреве
    
    # 📊 АУГМЕНТАЦИЯ ДАННЫХ
    augment=True,        # Применять аугментацию
    hsv_h=0.015,         # Аугментация Hue (оттенок)
    hsv_s=0.7,           # Аугментация Saturation (насыщенность)
    hsv_v=0.4,           # Аугментация Value (яркость)
    degrees=0.0,         # Вращение изображения (± градусы)
    translate=0.1,       # Смещение изображения (± доля)
    scale=0.5,           # Масштабирование изображения (± коэффициент)
    shear=0.0,           # Наклон изображения (± градусы)
    perspective=0.0,     # Перспектива изображения (± доля)
    flipud=0.0,          # Отражение по вертикали (вероятность)
    fliplr=0.5,          # Отражение по горизонтали (вероятность)
    
    # 🎛️ ДОПОЛНИТЕЛЬНЫЕ НАСТРОЙКИ
    patience=10,         # Терпение для ранней остановки
    save=True,           # Сохранять чекпоинты обучения
    save_period=-1,      # Сохранять чекпоинт каждые X эпох
    workers=8,           # Максимальное число workers для загрузки данных
    project='wildfire_classification',  # Имя проекта
    name='exp',          # Имя эксперимента
    exist_ok=False,      # Разрешить существующий проект/имя, не инкрементировать
    pretrained=True,     # Использовать предобученную модель
    optimizer='auto',    # Оптимизатор: [SGD, Adam, AdamW, RMSProP]
    verbose=True,        # Выводить подробный лог
    seed=0,              # Seed для воспроизводимости
    deterministic=True,  # Включить детерминистический режим
    single_cls=False,    # Обучать как датасет с одним классом
    image_weights=False, # Использовать взвешенный выбор изображений
    rect=False,          # Поддержка прямоугольного обучения
    cos_lr=False,        # Использовать косинусный scheduler для learning rate
    close_mosaic=10,     # Отключить мозаичную аугментацию для последних эпох
    
    # 💾 СИСТЕМНЫЕ НАСТРОЙКИ
    amp=True,            # Automatic Mixed Precision (AMP) обучение
    overlap_mask=True,   # Перекрывающиеся маски (для сегментации)
    mask_ratio=4,        # Коэффициент уменьшения маски (для сегментации)
    dropout=0.0,         # Dropout регуляризация (для классификации)
    val=True,            # Валидация/тестирование во время обучения
)
'''

"\nПроверенные параметры для YOLO-CLS:\nresults = model.train(\n    # 🎯 ОСНОВНЫЕ\n    data=IMAGE_PATH,\n    epochs=10,\n    imgsz=224,\n    batch=16,\n    device=0,\n\n    # 🔧 ОПТИМИЗАТОР И LEARNING RATE\n    lr0=0.01,           # Начальный learning rate\n    lrf=0.01,           # Финальный learning rate (lr0 * lrf)\n    momentum=0.937,      # SGD momentum\n    weight_decay=0.0005, # L2 регуляризация\n    warmup_epochs=3.0,   # Эпохи разогрева\n    warmup_momentum=0.8, # Momentum при разогреве\n    warmup_bias_lr=0.1,  # Learning rate для bias при разогреве\n\n    # 📊 АУГМЕНТАЦИЯ ДАННЫХ\n    augment=True,        # Применять аугментацию\n    hsv_h=0.015,         # Аугментация Hue (оттенок)\n    hsv_s=0.7,           # Аугментация Saturation (насыщенность)\n    hsv_v=0.4,           # Аугментация Value (яркость)\n    degrees=0.0,         # Вращение изображения (± градусы)\n    translate=0.1,       # Смещение изображения (± доля)\n    scale=0.5,           # Масштабирование изображения (± ко

In [4]:
from ultralytics import YOLO
import os
from ray import tune

IMAGE_PATH = "/home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset"

model = YOLO('yolov8n-cls.pt')
#model = YOLO('yolov8s-cls.pt')

# Явно укажите MLflow путь
#os.environ['MLFLOW_TRACKING_URI'] = 'file:./mlruns'

In [17]:
# Очистите датасет от скрытых файлов

import os
import shutil

def clean_dataset(dataset_path):
    """Удаляет скрытые файлы и папки из датасета"""
    for root, dirs, files in os.walk(dataset_path):
        # Удаляем скрытые папки
        for dir_name in dirs[:]:  # копируем список для безопасного удаления
            if dir_name.startswith('.') or dir_name.startswith('__'):
                full_path = os.path.join(root, dir_name)
                print(f"Удаляю скрытую папку: {full_path}")
                shutil.rmtree(full_path)
                dirs.remove(dir_name)  # убираем из обхода
        
        # Удаляем скрытые файлы
        for file_name in files:
            if file_name.startswith('.') or file_name.startswith('__'):
                full_path = os.path.join(root, file_name)
                print(f"Удаляю скрытый файл: {full_path}")
                os.remove(full_path)

# Очистите датасет
clean_dataset('/home/oleg/projects/ML/Simpsons/dataset/testset/testset')
clean_dataset('/home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset')

Удаляю скрытую папку: /home/oleg/projects/ML/Simpsons/dataset/testset/testset/.ipynb_checkpoints


In [7]:
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')

# Tuning конфиг
tune_config = {
    'lr0': (1e-5, 1e-2),           # диапазон для поиска
    'weight_decay': (1e-5, 1e-3),  # диапазон для поиска
}

# Запускаем TUNING
results = model.tune(
    data=IMAGE_PATH,
    epochs=5,
    iterations=10,
    space=tune_config,              # передаем конфиг
    batch=128,
    degrees=5.0,
    translate=0.1,
    fliplr=0.5,
    hsv_s=0.5,
    name='yolov8n_tuning',
    project='simpsons_classification'
)

# /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/weights/best.pt
# lr0: 0.00971
# weight_decay: 0.00052

Tuner: Initialized Tuner instance with 'tune_dir=/home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/10 with hyperparameters: {'lr0': 0.01, 'weight_decay': 0.0005}
New https://pypi.org/project/ultralytics/8.3.227 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.11.14 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flip

2025/11/12 13:46:10 INFO mlflow.tracking.fluent: Experiment with name 'simpsons_classification' does not exist. Creating a new experiment.


MLflow: logging run_id(abf6991e031f498fbd133e47352bf7b1) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 224 train, 224 val
Using 8 dataloader workers
Logging results to /home/oleg/projects/ML/Simpsons/simpsons_classification/train
Starting training for 5 epochs...

      Epoch    GPU_mem       loss  Instances       Size
        1/5      1.39G      3.335         24        224: 100% ━━━━━━━━━━━━ 158/158 10.8it/s 14.7s0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 30/30 35.4it/s 0.8s0.0s
                   all      0.397      0.749

      Epoch    GPU_mem       loss  Instances       Size
        2/5      1.64G      1.887         24        224: 100% ━━━━━━━━━━━━ 158/158 20.3it/s 7.8s0.1s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 30/30 38.8it/s 0.8s0.1s
                   all      0.815      0.914

      Epoch    GPU_mem       l

Premature end of JPEG file


Split complete in /home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset_split ✅
train: /home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset_split/train... found 20120 images in 42 classes ✅ 
val: /home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset_split/val... found 7597 images in 42 classes ✅ 
test: None...
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 30/30 33.0it/s 0.9s0.1s
                   all       0.92      0.977
Speed: 0.0ms preprocess, 0.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/oleg/projects/ML/Simpsons/simpsons_classification/train
MLflow: results logged to runs/mlflow
MLflow: disable with 'yolo settings mlflow=False'
💡 Learn more at https://docs.ultralytics.com/modes/train


...

Saved /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/tune_scatter_plots.png
Saved /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/tune_fitness.png

Tuner: 7/10 iterations complete ✅ (429.24s)
Tuner: Results saved to /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning
Tuner: Best fitness=0.9486 observed at iteration 6
Tuner: Best fitness metrics are {'metrics/accuracy_top1': 0.91997, 'metrics/accuracy_top5': 0.97723, 'val/loss': 0.32369, 'fitness': 0.9486}
Tuner: Best fitness model is /home/oleg/projects/ML/Simpsons/simpsons_classification/train
Printing '/home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/best_hyperparameters.yaml'

lr0: 0.00971
weight_decay: 0.00052

Tuner: Starting iteration 8/10 with hyperparameters: {'lr0': 0.00971, 'weight_decay': 0.00048}
New https://pypi.org/project/ultralytics/8.3.227 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.11.14 torc

Exception in thread Thread-3 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/oleg/miniconda3/envs/tensflow/lib/python3.11/threading.py", line 1045, in _bootstrap_inner


Saved /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/tune_scatter_plots.png
Saved /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/tune_fitness.png

Tuner: 9/10 iterations complete ✅ (550.48s)
Tuner: Results saved to /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning
Tuner: Best fitness=0.9486 observed at iteration 6
Tuner: Best fitness metrics are {'metrics/accuracy_top1': 0.91997, 'metrics/accuracy_top5': 0.97723, 'val/loss': 0.32369, 'fitness': 0.9486}
Tuner: Best fitness model is /home/oleg/projects/ML/Simpsons/simpsons_classification/train
Printing '/home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/best_hyperparameters.yaml'

lr0: 0.00971
weight_decay: 0.00052

Tuner: Starting iteration 10/10 with hyperparameters: {'lr0': 0.00832, 'weight_decay': 0.00039}
New https://pypi.org/project/ultralytics/8.3.227 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.11.14 tor

Exception in thread Thread-3 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/oleg/miniconda3/envs/tensflow/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/home/oleg/miniconda3/envs/tensflow/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/home/oleg/miniconda3/envs/tensflow/lib/python3.11/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/oleg/miniconda3/envs/tensflow/lib/python3.11/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/oleg/miniconda3/envs/tensflow/lib/python3.11/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/oleg/miniconda3/envs/tensflow/lib/python3.11/site-packages

Saved /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/tune_scatter_plots.png
Saved /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/tune_fitness.png

Tuner: 10/10 iterations complete ✅ (611.09s)
Tuner: Results saved to /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning
Tuner: Best fitness=0.9486 observed at iteration 6
Tuner: Best fitness metrics are {'metrics/accuracy_top1': 0.91997, 'metrics/accuracy_top5': 0.97723, 'val/loss': 0.32369, 'fitness': 0.9486}
Tuner: Best fitness model is /home/oleg/projects/ML/Simpsons/simpsons_classification/train
Printing '/home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/best_hyperparameters.yaml'

lr0: 0.00971
weight_decay: 0.00052



In [10]:
# YOLO автоматически просканирует папку и поймет структуру

# /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/weights/best.pt
# lr0: 0.00971
# weight_decay: 0.00052

model = YOLO('/home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/weights/best.pt')

results = model.train(
    data=IMAGE_PATH,  # ✅ ВАШ ПУТЬ ЗДЕСЬ
    # Сбалансированная аугментация
    degrees=5.0,
    translate=0.1,
    fliplr=0.5,          # Отражение по горизонтали (вероятность)
    hsv_s=0.5,           # Аугментация Saturation (насыщенность)
    
    epochs=20,
    imgsz=224,
    batch=128,
    lr0=0.00971,
    weight_decay=0.00052,
    patience=10,
    save=True,
    device=0,
    plots=True,        # Создает подробные графики
    
    # Для лучшего отслеживания в MLflow
    name='yolov8_experiment_simpsons',
    project='YOLO-simpsons',
    
    verbose=True,
    val=True,            # Валидация/тестирование во время обучения
)

# Default params = 
# classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 30/30 18.5it/s -1.7s.0s
#    all      0.988 


# lr0: 0.00971
# weight_decay: 0.00052
# top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 30/30 23.5it/s 1.3s0.1s
#                    all      0.992 



New https://pypi.org/project/ultralytics/8.3.227 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.221 🚀 Python-3.11.14 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00971, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/oleg/projects/ML/Simpsons/simpsons_classific

In [ ]:
cd /home/oleg/projects/ML/Wildfire_Satelite && mlflow server --backend-store-uri runs/mlflow --host 0.0.0.0 --port 5000

In [11]:
# Дообученная модель top1_acc - 0.992
/home/oleg/projects/ML/Simpsons/YOLO-simpsons/yolov8_experiment_simpsons2/weights/best.pt

In [ ]:
# /home/oleg/projects/ML/Simpsons/simpsons_classification/yolov8n_tuning/weights/best.pt
# lr0: 0.00971
# weight_decay: 0.00052

In [12]:
# Загружаем обученную модель
model = YOLO('/home/oleg/projects/ML/Simpsons/YOLO-simpsons/yolov8_experiment_simpsons2/weights/best.pt')

# Смотрим классы
print("🎯 Классы модели:")
print(f"Количество классов: {model.names}")
print(f"Словарь классов: {model.names}")
print(f"ID -> Имя: {dict(enumerate(model.names))}")

# Альтернативные способы
print(f"\n📋 Все атрибуты модели:")
print(f"model.names: {model.names}")
print(f"model.model.names: {getattr(model.model, 'names', 'Not found')}")

🎯 Классы модели:
Количество классов: {0: 'abraham_grampa_simpson', 1: 'agnes_skinner', 2: 'apu_nahasapeemapetilon', 3: 'barney_gumble', 4: 'bart_simpson', 5: 'carl_carlson', 6: 'charles_montgomery_burns', 7: 'chief_wiggum', 8: 'cletus_spuckler', 9: 'comic_book_guy', 10: 'disco_stu', 11: 'edna_krabappel', 12: 'fat_tony', 13: 'gil', 14: 'groundskeeper_willie', 15: 'homer_simpson', 16: 'kent_brockman', 17: 'krusty_the_clown', 18: 'lenny_leonard', 19: 'lionel_hutz', 20: 'lisa_simpson', 21: 'maggie_simpson', 22: 'marge_simpson', 23: 'martin_prince', 24: 'mayor_quimby', 25: 'milhouse_van_houten', 26: 'miss_hoover', 27: 'moe_szyslak', 28: 'ned_flanders', 29: 'nelson_muntz', 30: 'otto_mann', 31: 'patty_bouvier', 32: 'principal_skinner', 33: 'professor_john_frink', 34: 'rainier_wolfcastle', 35: 'ralph_wiggum', 36: 'selma_bouvier', 37: 'sideshow_bob', 38: 'sideshow_mel', 39: 'snake_jailbird', 40: 'troy_mcclure', 41: 'waylon_smithers'}
Словарь классов: {0: 'abraham_grampa_simpson', 1: 'agnes_skin

In [13]:
from IPython.display import Image, display
import os

results_dir = "/home/oleg/projects/ML/Simpsons/YOLO-simpsons/yolov8_experiment_simpsons2/weights/best.pt"

print("📈 Готовые графики из папки обучения:")

# Показываем results.png
results_path = os.path.join(results_dir, 'results.png')
if os.path.exists(results_path):
    print("✅ results.png: Общие метрики обучения")
    display(Image(filename=results_path, width=1000))

# Показываем confusion matrix
cm_path = os.path.join(results_dir, 'confusion_matrix.png')
if os.path.exists(cm_path):
    print("✅ confusion_matrix.png: Матрица ошибок")
    display(Image(filename=cm_path, width=800))

# Показываем примеры изображений
train_batch_path = os.path.join(results_dir, 'train_batch0.jpg')
if os.path.exists(train_batch_path):
    print("✅ train_batch0.jpg: Примеры обучающих изображений")
    display(Image(filename=train_batch_path, width=800))

📈 Готовые графики из папки обучения:


In [14]:
from ultralytics import YOLO

# Загружаем модель
model = YOLO('/home/oleg/projects/ML/Simpsons/YOLO-simpsons/yolov8_experiment_simpsons2/weights/best.pt')

# Тестируем на тестовых данных
results = model.val(data=IMAGE_PATH, split='test', verbose=True)

print("\n" + "="*50)
print("📊 ОСНОВНЫЕ МЕТРИКИ")
print("="*50)
print(f"🎯 Top-1 Accuracy: {results.top1:.4f}")
print(f"📈 Top-5 Accuracy: {results.top5:.4f}")
print("="*50)

Ultralytics 8.3.221 🚀 Python-3.11.14 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,488,682 parameters, 0 gradients, 3.3 GFLOPs
WARNING ⚠️ Dataset 'split=train' not found at /home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset/train
Found 20933 images in subdirectories. Attempting to split...
Splitting /home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset (42 classes, 20933 images) into 80% train, 20% val...
Split complete in /home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset_split ✅
WARNING ⚠️ Dataset 'split=test' not found, using 'split=val' instead.
train: /home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset_split/train... found 20778 images in 42 classes ✅ 
val: /home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset_split/val... found 10276 images in 42 classes ✅ 
test: /home/oleg/projects/ML/Simpsons/dataset/train/simpsons_dataset_split/val... found 10276 images in 42 classes ✅

In [20]:
from ultralytics import YOLO
import pandas as pd
import os
from pathlib import Path
import glob

def create_submission_csv(results, csv_path, test_data_path):
    """Создает CSV файл с предсказаниями в формате для submission"""
    
    predictions = []
    
    for result in results:
        # Получаем имя файла
        img_path = Path(result.path)
        filename = img_path.name
        
        # Получаем предсказания
        if hasattr(result, 'probs'):
            # Для classification моделей
            top1_idx = result.probs.top1
            top1_conf = result.probs.top1conf.cpu().numpy()
            top5_indices = result.probs.top5
            top5_confidences = result.probs.top5conf.cpu().numpy()
            
            # Получаем имя класса
            if hasattr(result, 'names'):
                class_name = result.names[top1_idx]
            else:
                class_name = f"class_{top1_idx}"
            
            pred_row = {
                'image_id': filename,
                'predicted_class': class_name,
                'predicted_class_id': top1_idx,
                'confidence': top1_conf,
                'top5_classes': ' '.join([str(x) for x in top5_indices]),
                'top5_confidences': ' '.join([f"{x:.4f}" for x in top5_confidences])
            }
        else:
            # Для detection моделей или если нет probs
            pred_row = {
                'image_id': filename,
                'predicted_class': 'unknown',
                'predicted_class_id': -1,
                'confidence': 0,
                'top5_classes': 'N/A',
                'top5_confidences': 'N/A'
            }
        
        predictions.append(pred_row)
    
    # Создаем DataFrame
    pred_df = pd.DataFrame(predictions)
    
    # Сохраняем в CSV
    output_path = csv_path.replace('.csv', '_predictions.csv')
    pred_df.to_csv(output_path, index=False)
    print(f"💾 Predictions saved to: {output_path}")
    print(f"📊 Всего предсказаний: {len(pred_df)}")
    
    return pred_df

# Основной код
model = YOLO('/home/oleg/projects/ML/Simpsons/YOLO-simpsons/yolov8_experiment_simpsons2/weights/best.pt')

test_data = '/home/oleg/projects/ML/Simpsons/dataset/testset/testset'
csv_path = '/home/oleg/projects/ML/Simpsons/dataset/sample_submission.csv'

print("🔍 Проверка структуры тестовых данных...")
if os.path.exists(test_data):
    items = os.listdir(test_data)
    print(f"Элементы в testset: {len(items)}")
    
    # Проверим есть ли подпапки с классами
    subdirs = [d for d in items if os.path.isdir(os.path.join(test_data, d))]
    if subdirs:
        print(f"Найдены подпапки-классы: {subdirs[:5]}...")  # первые 5
        # Если есть подпапки - используем model.val()
        results = model.val(data=test_data, verbose=True)
    else:
        print("✅ Нет подпапок с классами. Используем model.predict()")
        # Получаем все изображения
        image_files = glob.glob(os.path.join(test_data, "*.jpg")) + \
                     glob.glob(os.path.join(test_data, "*.jpeg")) + \
                     glob.glob(os.path.join(test_data, "*.png"))
        print(f"Найдено изображений: {len(image_files)}")
        
        # Предсказания для всех изображений
        results = model.predict(image_files, save=False, verbose=True)
        
        # Создаем CSV с предсказаниями
        submission_df = create_submission_csv(results, csv_path, test_data)

print("\n" + "="*50)
print("📊 РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ")
print("="*50)

# Если использовали model.val() - выводим метрики
if hasattr(results, 'top1'):
    print(f"🎯 Top-1 Accuracy: {results.top1:.4f}")
    print(f"📈 Top-5 Accuracy: {results.top5:.4f}")
    print(f"📊 Speed: {results.speed}")
    
    # Выводим все доступные атрибуты
    print(f"\n🔍 Все доступные атрибуты:")
    for attr in dir(results):
        if not attr.startswith('_'):
            try:
                value = getattr(results, attr)
                if not callable(value):
                    print(f"  {attr}: {value}")
            except:
                pass
else:
    # Если использовали predict - выводим статистику предсказаний
    print(f"✅ Обработано изображений: {len(results)}")
    
    # Пример предсказания
    if len(results) > 0 and hasattr(results[0], 'probs'):
        first_result = results[0]
        print(f"📝 Пример предсказания для {Path(first_result.path).name}:")
        print(f"  Топ-1 класс: {first_result.names[first_result.probs.top1]}")
        print(f"  Уверенность: {first_result.probs.top1conf:.4f}")
        print(f"  Топ-5 классов: {[first_result.names[i] for i in first_result.probs.top5]}")

print("="*50)

# Дополнительная статистика если использовали predict
if 'submission_df' in locals():
    print(f"\n📈 СТАТИСТИКА ПРЕДСКАЗАНИЙ:")
    print(f"📊 Уникальных классов предсказано: {submission_df['predicted_class'].nunique()}")
    print(f"📈 Средняя уверенность: {submission_df['confidence'].mean():.4f}")
    print(f"🎯 Максимальная уверенность: {submission_df['confidence'].max():.4f}")
    print(f"📉 Минимальная уверенность: {submission_df['confidence'].min():.4f}")
    
    # Топ предсказанных классов
    class_counts = submission_df['predicted_class'].value_counts()
    print(f"\n🏆 Топ-5 самых частых предсказаний:")
    for class_name, count in class_counts.head().items():
        print(f"  {class_name}: {count} раз")

🔍 Проверка структуры тестовых данных...
Элементы в testset: 991
✅ Нет подпапок с классами. Используем model.predict()
Найдено изображений: 991

0: 224x224 moe_szyslak 0.99, charles_montgomery_burns 0.00, gil 0.00, principal_skinner 0.00, agnes_skinner 0.00, 0.1ms
1: 224x224 principal_skinner 0.29, charles_montgomery_burns 0.16, marge_simpson 0.12, edna_krabappel 0.12, lisa_simpson 0.05, 0.1ms
2: 224x224 lenny_leonard 1.00, moe_szyslak 0.00, charles_montgomery_burns 0.00, cletus_spuckler 0.00, ned_flanders 0.00, 0.1ms
3: 224x224 apu_nahasapeemapetilon 1.00, lenny_leonard 0.00, ned_flanders 0.00, lisa_simpson 0.00, charles_montgomery_burns 0.00, 0.1ms
4: 224x224 abraham_grampa_simpson 1.00, homer_simpson 0.00, lenny_leonard 0.00, lisa_simpson 0.00, bart_simpson 0.00, 0.1ms
5: 224x224 principal_skinner 1.00, agnes_skinner 0.00, moe_szyslak 0.00, charles_montgomery_burns 0.00, waylon_smithers 0.00, 0.1ms
6: 224x224 bart_simpson 1.00, waylon_smithers 0.00, lisa_simpson 0.00, homer_simpson 0